## オリジナル

* https://saiensu.co.jp/search/?isbn=978-4-7819-1678-1&y=2026#support
  * code19_chap6_reidai_confidence_interval_code_Julia.txt

In [1]:
# パッケージがインストール済なら次の行は不要
# import Pkg; Pkg.add(["Distributions"])
using Distributions
n = 300
pvar = 210 / 300
# 二項検定の実行結果から信頼区間の項目を取得
I = quantile(Binomial(n, pvar), [0.025, 0.975])
println("pの95%信頼区間：", [I[1]/n, I[2]/n])

pの95%信頼区間：[0.6466666666666666, 0.75]


これだと「二項検定の実行結果から信頼区間の項目を取得」になっていない。

ブートストラップ法で求めた区間推定の計算になってしまっている。

## 訂正案

In [2]:
# パッケージがインストール済なら次の行は不要
# import Pkg; Pkg.add(["HypothesisTests"])
using HypothesisTests
n = 300
x = 210
level = 0.95
method = :clopper_pearson
# 二項検定の実行結果から信頼区間の項目を取得
result = BinomialTest(x, n)
println("pの$(round(Int, 100level))%信頼区間：", confint(result; level))

pの95%信頼区間：(0.6446804955369393, 0.7513184234759531)


## さらなる訂正案

普通はP値にも触れる。

In [3]:
# パッケージがインストール済なら次の行は不要
# import Pkg; Pkg.add(["HypothesisTests"])
using HypothesisTests
n = 300
x = 210
level = 0.95
p_test = 0.6446804955369393
# 二項検定の実行結果から信頼区間の項目を取得
result = BinomialTest(x, n, p_test)
println("pの$(round(Int, 100level))%信頼区間：", confint(result; level))
println("仮説p=$(p_test)の両側P値：", pvalue(result))

pの95%信頼区間：(0.6446804955369393, 0.7513184234759531)
仮説p=0.6446804955369393の両側P値：0.04999999999999973


In [4]:
?BinomialTest

search: BinomialTest Binomial MultinomialLRTest binomial @invokelatest



```julia
BinomialTest(x::Integer, n::Integer, p::Real = 0.5)
BinomialTest(x::AbstractVector{Bool}, p::Real = 0.5)
```

Perform a binomial test of the null hypothesis that the distribution from which `x` successes were encountered in `n` draws (or alternatively from which the vector `x` was drawn) has success probability `p` against the alternative hypothesis that the success probability is not equal to `p`.

Computed confidence intervals by default are Clopper-Pearson intervals. See the [`confint(::BinomialTest)`](@ref) documentation for a list of supported methods to compute confidence intervals.

Implements: [`pvalue`](@ref), [`confint(::BinomialTest)`](@ref)


In [5]:
?confint(::BinomialTest)

```julia
confint(test::BinomialTest; level = 0.95, tail = :both, method = :clopper_pearson)
```

Compute a confidence interval with coverage `level` for a binomial proportion using one of the following methods. Possible values for `method` are:

  * `:clopper_pearson` (default): Clopper-Pearson interval is based on the binomial distribution. The empirical coverage is never less than the nominal coverage of `level`; it is usually too conservative.
  * `:wald`: Wald (or normal approximation) interval relies on the standard approximation of the actual binomial distribution by a normal distribution. Coverage can be erratically poor for success probabilities close to zero or one.
  * `:waldcc`: Wald interval with a continuity correction that extends the interval by `1/2n` on both ends.
  * `:wilson`: Wilson score interval relies on a normal approximation. In contrast to `:wald`, the standard deviation is not approximated by an empirical estimate, resulting in good empirical coverages even for small numbers of draws and extreme success probabilities.
  * `:jeffrey`: Jeffreys interval is a Bayesian credible interval obtained by using a non-informative Jeffreys prior. The interval is very similar to the Wilson interval.
  * `:agresti_coull`: Agresti-Coull interval is a simplified version of the Wilson interval; both are centered around the same value. The Agresti Coull interval has higher or equal coverage.
  * `:arcsine`: Confidence interval computed using the arcsine transformation to make $var(p)$ independent of the probability $p$.

# References

  * [Brown, Cai, and DasGupta (2001). Interval estimation for a binomial proportion](@cite brown2001)
  * [Pires and Amado (2008). Interval Estimators for a Binomial Proportion:  Comparison of Twenty Methods](@cite pires2008)

# External links

  * [Binomial confidence interval on Wikipedia](https://en.wikipedia.org/wiki/ Binomial_proportion_confidence_interval)


In [6]:
?pvalue

search: pvalue values parse pvar eval evalfile scale



```julia
pvalue(test)
```

Compute the p-value for a given significance test.

---

```julia
pvalue(x::FisherExactTest; tail = :both, method = :central)
```

Compute the p-value for a given Fisher exact test.

The one-sided p-values are based on Fisher's non-central hypergeometric distribution $f_ω(i)$ with odds ratio $ω$:

$$
    \begin{align*}
        p_ω^{(\text{left})} &=\sum_{i ≤ a} f_ω(i)\\
        p_ω^{(\text{right})} &=\sum_{i ≥ a} f_ω(i)
    \end{align*}
$$

For `tail = :both`, possible values for `method` are:

  * `:central` (default): Central interval, i.e. the p-value is two times the minimum of the one-sided p-values.
  * `:minlike`: Minimum likelihood interval, i.e. the p-value is computed by summing all tables with the same marginals that are equally or less probable:

    $$
        p_ω = \sum_{f_ω(i)≤ f_ω(a)} f_ω(i)
    $$

# References

  * [Gibbons and Pratt (1975). P-Values: Interpretation and Methodology](@cite gibbons1975)
  * [Fay (2010). Confidence Intervals That Match Fisher's Exact or Blaker's  Exact Tests](@cite fay2010)
